# Advanced Prompting Techniques

**Module:** 07-prompt-engineering

**Notebook:** `03-advanced-prompting-techniques.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Why Advanced Techniques?** with clear contracts and failure modes
- Explain and apply **Chain-of-Thought (CoT)** with clear contracts and failure modes
- Explain and apply **Tree of Thoughts (ToT)** with clear contracts and failure modes
- Explain and apply **Graph of Thought** with clear contracts and failure modes
- Explain and apply **Self-Consistency** with clear contracts and failure modes
- Explain and apply **ReAct (Reason + Act)** with clear contracts and failure modes
- Explain and apply **Reflection** with clear contracts and failure modes
- Explain and apply **Decomposition** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Advanced Prompting Techniques

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Why Advanced Techniques?**
2. **Chain-of-Thought (CoT)**
3. **Tree of Thoughts (ToT)**
4. **Graph of Thought**
5. **Self-Consistency**
6. **ReAct (Reason + Act)**
7. **Reflection**
8. **Decomposition**
9. **Step-Back Prompting**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Why Advanced Techniques?

### Definition
**Why Advanced Techniques?** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Why Advanced Techniques? typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Why Advanced Techniques?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Why Advanced Techniques? as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Why Advanced Techniques? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Why Advanced Techniques?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Why Advanced Techniques? when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Why Advanced Techniques? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Why Advanced Techniques?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Why Advanced Techniques?"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Why Advanced Techniques?"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Why Advanced Techniques?"}
strong = {"definition": "Why Advanced Techniques?", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Why Advanced Techniques?"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Why Advanced Techniques?", "passed": len(checks)-len(failed), "failed": failed})


## Chain-of-Thought (CoT)

### Definition
**Chain-of-Thought** prompting asks the model to produce intermediate reasoning steps before a final answer.

### Why it matters
Multi-step tasks fail when the model skips constraints; visible/latent steps reduce that.

### How it works
Request numbered reasoning and a delimited final answer; consider hiding scratchpads from end users.

### Intuition
Show your work—same reason as math class.

### Pitfalls
- Plausible-but-wrong steps
- Token cost on trivial tasks
- Leaking private reasoning

### When to use
Logic-heavy tasks; skip for simple extraction.


In [ ]:
# Demo: make "Chain-of-Thought (CoT)" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Chain-of-Thought (CoT)"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


### Worked scenario — Chain-of-Thought (CoT)

**Situation:** A team wants to productionize a feature involving **Chain-of-Thought (CoT)**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Tree of Thoughts (ToT)

### Definition
**Tree of Thoughts (ToT)** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Tree of Thoughts (ToT) typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tree of Thoughts (ToT): (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Tree of Thoughts (ToT) as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tree of Thoughts (ToT) as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tree of Thoughts (ToT)
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Tree of Thoughts (ToT) when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tree of Thoughts (ToT)" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tree of Thoughts (ToT)"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Tree of Thoughts (ToT)"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Tree of Thoughts (ToT)"}
strong = {"definition": "Tree of Thoughts (ToT)", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Tree of Thoughts (ToT)"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Tree of Thoughts (ToT)", "passed": len(checks)-len(failed), "failed": failed})


## Graph of Thought

### Definition
**Graph of Thought** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Graph of Thought typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Graph of Thought: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Graph of Thought as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Graph of Thought as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Graph of Thought
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Graph of Thought when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Graph of Thought" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Graph of Thought"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


### Worked scenario — Graph of Thought

**Situation:** A team wants to productionize a feature involving **Graph of Thought**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Self-Consistency

### Definition
**Self-Consistency** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Self-Consistency typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Self-Consistency: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Self-Consistency as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Self-Consistency as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Self-Consistency
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Self-Consistency when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Self-Consistency" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Self-Consistency"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Self-Consistency"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Self-Consistency"}
strong = {"definition": "Self-Consistency", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Self-Consistency"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Self-Consistency", "passed": len(checks)-len(failed), "failed": failed})


## ReAct (Reason + Act)

### Definition
**ReAct (Reason + Act)** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around ReAct (Reason + Act) typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For ReAct (Reason + Act): (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain ReAct (Reason + Act) as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating ReAct (Reason + Act) as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for ReAct (Reason + Act)
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use ReAct (Reason + Act) when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "ReAct (Reason + Act)" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "ReAct (Reason + Act)"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — ReAct (Reason + Act)

**Situation:** A team wants to productionize a feature involving **ReAct (Reason + Act)**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Reflection

### Definition
**Reflection** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Reflection typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Reflection: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Reflection as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Reflection as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Reflection
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Reflection when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Reflection" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Reflection"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Reflection"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Reflection"}
strong = {"definition": "Reflection", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Reflection"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Reflection", "passed": len(checks)-len(failed), "failed": failed})


## Decomposition

### Definition
**Decomposition** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Decomposition typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Decomposition: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Decomposition as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Decomposition as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Decomposition
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Decomposition when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Decomposition" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Decomposition"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_7 = ConceptContract()
print(json.dumps({"contract": asdict(contract_7), "health": contract_7.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Decomposition"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Decomposition"}
strong = {"definition": "Decomposition", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Decomposition"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Decomposition", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Decomposition

**Situation:** A team wants to productionize a feature involving **Decomposition**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Step-Back Prompting

### Definition
**Step-Back Prompting** is a core building block in 03-advanced-prompting-techniques within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Step-Back Prompting typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Step-Back Prompting: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Step-Back Prompting as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Step-Back Prompting as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Step-Back Prompting
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Step-Back Prompting when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Step-Back Prompting" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Step-Back Prompting"
    notebook: str = "03-advanced-prompting-techniques"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_8 = ConceptContract()
print(json.dumps({"contract": asdict(contract_8), "health": contract_8.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Step-Back Prompting"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Step-Back Prompting"}
strong = {"definition": "Step-Back Prompting", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Step-Back Prompting"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Step-Back Prompting", "passed": len(checks)-len(failed), "failed": failed})


## Comparison Snapshot

Use this table when reviewing designs in **Advanced Prompting Techniques**.

| Topic | Do | Don't |
|-------|----|-------|
| Why Advanced Techniques? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Chain-of-Thought (CoT) | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tree of Thoughts (ToT) | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Graph of Thought | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Self-Consistency | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| ReAct (Reason + Act) | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Why Advanced Techniques? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Chain-of-Thought (CoT) | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tree of Thoughts (ToT) | Key concept covered in this notebook; see its section for definition and pitfalls |
| Graph of Thought | Key concept covered in this notebook; see its section for definition and pitfalls |
| Self-Consistency | Key concept covered in this notebook; see its section for definition and pitfalls |
| ReAct (Reason + Act) | Key concept covered in this notebook; see its section for definition and pitfalls |
| Reflection | Key concept covered in this notebook; see its section for definition and pitfalls |
| Decomposition | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Advanced Prompting Techniques** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **Why Advanced Techniques?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Chain-of-Thought (CoT)**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Tree of Thoughts (ToT)**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Graph of Thought**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Self-Consistency**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
